<div style="background: linear-gradient(135deg, #8E2DE2 0%, #4A00E0 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 4</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Replication and ISR (In-Sync Replicas)</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Discover how Kafka achieves high availability and fault tolerance through data replication.</p>
</div>

---

## 🎯 Overview

Kafka is designed to be highly available and resilient to hardware failures. It achieves this by **replicating** topic partitions across multiple brokers.

Key concepts you will explore:
- 👯 **Replication Factor:** How many copies of the data exist in the cluster.
- 👑 **Leader:** The single broker responsible for all reads and writes for a specific partition.
- 🛡️ **Followers:** Brokers that passively copy data from the Leader.
- ✅ **ISR (In-Sync Replicas):** The subset of replicas that are fully caught up with the Leader. If the Leader dies, only a broker in the ISR can take its place.

---

## ⚙️ Prerequisites

For this lab, our single-node cluster from previous labs is not enough. **We need a multi-broker cluster.** 
We have provided a specific `docker-compose.yml` in this `Lab4` directory that spins up a **3-Node Kafka Cluster**.

<div style="background-color: rgba(243, 156, 18, 0.1); border-left: 4px solid #f39c12; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>⚠️ Important:</strong> Make sure you have stopped the cluster from Lab 1 before proceeding. Run <code>docker-compose down</code> in the root folder if you haven't already.
</div>

---

## <span style="color: #8E2DE2;">Step 1:</span> Start the 3-Node Kafka Cluster

Let's spin up our new cluster. It contains `zookeeper`, `kafka1`, `kafka2`, and `kafka3`.

In [10]:
!docker-compose -f ../../docker-compose.yml down
!docker-compose up -d

time="2026-05-17T19:54:56+05:30" level=warning msg="d:\\trainings\\CCDAK-2\\kafka_training\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-05-17T19:54:56+05:30" level=warning msg="d:\\trainings\\CCDAK-2\\kafka_training\\Labs\\Lab3\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Network lab3_default  Creating
 Network lab3_default  Created
 Container zookeeper  Creating
 Container zookeeper  Created
 Container kafka3  Creating
 Container kafka1  Creating
 Container kafka2  Creating
 Container kafka1  Created
 Container kafka2  Created
 Container kafka3  Created
 Container zookeeper  Starting
 Container zookeeper  Started
 Container kafka3  Starting
 Container kafka2  Starting
 Container kafka1  Starting
 Container kafka3  Started
 Container kafka1  Started
 Container kafka2  Started


Check that all 3 Kafka brokers and Zookeeper are running:

In [8]:
!docker ps

CONTAINER ID   IMAGE                             COMMAND                  CREATED         STATUS         PORTS                                         NAMES
ba528f775ca1   confluentinc/cp-kafka:7.5.0       "/etc/confluent/dock…"   2 minutes ago   Up 7 seconds   0.0.0.0:9093->9093/tcp, [::]:9093->9093/tcp   lab3-kafka2-1
5195d1742156   confluentinc/cp-kafka:7.5.0       "/etc/confluent/dock…"   2 minutes ago   Up 7 seconds   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp   lab3-kafka1-1
865e3789665e   confluentinc/cp-kafka:7.5.0       "/etc/confluent/dock…"   2 minutes ago   Up 7 seconds   0.0.0.0:9094->9094/tcp, [::]:9094->9094/tcp   lab3-kafka3-1
58e533f76c18   confluentinc/cp-zookeeper:7.5.0   "/etc/confluent/dock…"   2 minutes ago   Up 8 seconds   0.0.0.0:2181->2181/tcp, [::]:2181->2181/tcp   lab3-zookeeper-1


---

## <span style="color: #8E2DE2;">Step 2:</span> Create a Replicated Topic

Now that we have 3 brokers, we can create a topic with a `replication-factor` of 3. We will name it `secure-orders`.

In [11]:
!docker exec kafka1 kafka-topics \
  --bootstrap-server localhost:9092 \
  --create \
  --topic secure-orders \
  --partitions 3 \
  --replication-factor 3

Created topic secure-orders.


---

## <span style="color: #8E2DE2;">Step 3:</span> Describe the Topic to see the ISR

Let's inspect the partition layout. Pay close attention to the output.

**What to look for:**
- **Leader:** This is the Broker ID (1, 2, or 3) currently handling traffic for this partition.
- **Replicas:** Lists all brokers that hold a copy of the partition (e.g., 1,2,3).
- **Isr:** Lists all brokers that are completely caught up and "in-sync" (e.g., 1,2,3). Right now, all replicas should be in the ISR.

In [12]:
!docker exec kafka1 kafka-topics \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic secure-orders

Topic: secure-orders	TopicId: JuTT1UUtTp68DcBC-MFcJA	PartitionCount: 3	ReplicationFactor: 3	Configs: 
	Topic: secure-orders	Partition: 0	Leader: 3	Replicas: 3,2,1	Isr: 3,2,1
	Topic: secure-orders	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 1,3,2
	Topic: secure-orders	Partition: 2	Leader: 2	Replicas: 2,1,3	Isr: 2,1,3


[2026-05-17 14:25:24,228] WARN [AdminClient clientId=adminclient-1] Connection to node 3 (localhost/127.0.0.1:9094) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-05-17 14:25:24,276] WARN [AdminClient clientId=adminclient-1] Connection to node 2 (localhost/127.0.0.1:9093) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-05-17 14:25:24,277] WARN [AdminClient clientId=adminclient-1] Connection to node 3 (localhost/127.0.0.1:9094) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)


---

## <span style="color: #8E2DE2;">Step 4:</span> Simulate a Broker Failure

Let's test Kafka's fault tolerance! Look at the `Leader` column from the previous step for **Partition 0**. Let's assume `kafka3` was the leader (or one of the ISRs).

We are going to brutally kill `kafka3`.

In [13]:
!docker stop kafka3

kafka3


---

## <span style="color: #8E2DE2;">Step 5:</span> Observe the ISR Shrink

Immediately describe the topic again.

<div style="background-color: rgba(63, 185, 80, 0.1); border-left: 4px solid #3fb950; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>✅ Observation:</strong><br><br>
  1. Notice that Broker 3 is completely missing from the <code>Isr</code> list! It shrunk to (1,2) or similar.<br>
  2. If Broker 3 was a <code>Leader</code> for any partition, notice that Kafka automatically performed a <strong>Leader Election</strong> and assigned a new leader from the surviving ISRs. Your data is perfectly safe and the cluster is still fully operational!
</div>

In [14]:
!docker exec kafka1 kafka-topics \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic secure-orders

Topic: secure-orders	TopicId: JuTT1UUtTp68DcBC-MFcJA	PartitionCount: 3	ReplicationFactor: 3	Configs: 
	Topic: secure-orders	Partition: 0	Leader: 2	Replicas: 3,2,1	Isr: 2,1
	Topic: secure-orders	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 1,2
	Topic: secure-orders	Partition: 2	Leader: 2	Replicas: 2,1,3	Isr: 2,1


[2026-05-17 14:25:57,725] WARN [AdminClient clientId=adminclient-1] Connection to node 2 (localhost/127.0.0.1:9093) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)


---

## <span style="color: #8E2DE2;">Step 6:</span> Bring the Broker Back Online

The operations team has fixed the server. Let's restart `kafka3`.

In [15]:
!docker start kafka3
import time
time.sleep(10) # Give it a moment to boot up and sync

kafka3


---

## <span style="color: #8E2DE2;">Step 7:</span> Observe the ISR Recover

Describe the topic one last time. 

Broker 3 has reconnected to the leaders, downloaded any messages it missed while it was dead, and once fully caught up, Zookeeper/KRaft adds it back into the `Isr` list.

In [16]:
!docker exec kafka1 kafka-topics \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic secure-orders

Topic: secure-orders	TopicId: JuTT1UUtTp68DcBC-MFcJA	PartitionCount: 3	ReplicationFactor: 3	Configs: 
	Topic: secure-orders	Partition: 0	Leader: 2	Replicas: 3,2,1	Isr: 2,1,3
	Topic: secure-orders	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 1,2,3
	Topic: secure-orders	Partition: 2	Leader: 2	Replicas: 2,1,3	Isr: 2,1,3


[2026-05-17 14:26:27,133] WARN [AdminClient clientId=adminclient-1] Connection to node 2 (localhost/127.0.0.1:9093) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)
[2026-05-17 14:26:27,150] WARN [AdminClient clientId=adminclient-1] Connection to node 3 (localhost/127.0.0.1:9094) could not be established. Broker may not be available. (org.apache.kafka.clients.NetworkClient)


---

## 🧹 Clean Up

Bring down the 3-node cluster and clean up the volumes.

In [17]:
!docker-compose down -v

time="2026-05-17T19:56:33+05:30" level=warning msg="d:\\trainings\\CCDAK-2\\kafka_training\\Labs\\Lab3\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container kafka3  Stopping
 Container kafka1  Stopping
 Container kafka2  Stopping
 Container kafka1  Stopped
 Container kafka1  Removing
 Container kafka1  Removed
 Container kafka2  Stopped
 Container kafka2  Removing
 Container kafka2  Removed
 Container kafka3  Stopped
 Container kafka3  Removing
 Container kafka3  Removed
 Container zookeeper  Stopping
 Container zookeeper  Stopped
 Container zookeeper  Removing
 Container zookeeper  Removed
 Network lab3_default  Removing
 Network lab3_default  Removed


<div style="background-color: rgba(142, 45, 226, 0.1); border: 1px solid rgba(142, 45, 226, 0.3); padding: 20px; text-align: center; border-radius: 8px; margin-top: 40px;">
  <h3 style="color: #8E2DE2; margin-bottom: 10px;">🎉 Lab 4 Complete!</h3>
  <p style="color: #8b949e; margin: 0;">You have successfully witnessed Kafka's fault tolerance in action! You saw how the ISR shrinks during failure to protect data integrity, and automatically expands during recovery.</p>
</div>